# Bases de données — Exercice 3 : les stations Vélib' (correction)

Ce notebook contient, pour chaque question, trois blocs **toujours séparés** :

| Bloc | Contenu |
|:---|:---|
| 📝 **ÉNONCÉ** | la question, recopiée du sujet |
| ✅ **RÉSOLUTION** | la réponse attendue (requête SQL, code et/ou texte) |
| 💡 **EXPLICATION PÉDAGOGIQUE** | pourquoi ça marche, le lien avec le cours, les erreurs à éviter. **N'est pas à recopier.** |

**Notions travaillées :** `SELECT ... WHERE`, `COUNT`, `AVG`, `LIKE`, `MIN`/`MAX` et requête imbriquée, `AND`, `ORDER BY` (`DESC`), jointure entre deux relations par une clé étrangère.

**Données :** le fichier `velib.sql` du professeur n'est pas disponible en ligne. Le fichier `velib.sql` placé à côté de ce notebook a été **reconstitué** à partir des **données ouvertes** officielles de Vélib' Métropole (flux GBFS public), avec exactement le schéma relationnel du sujet. C'est un instantané pris le **14 novembre 2026 à 2 h 18** (en pleine nuit) : les nombres obtenus ici seront **différents** de ceux de la classe, car les données changent toutes les minutes. **Les requêtes, elles, sont les mêmes.**

Exécuter les cellules **dans l'ordre** (*Cell → Run All*).

> Ces données obéissent au schéma relationnel ci-dessous.
> - Information(<u>id</u> : CHAR, name : CHAR, lat : REAL, lon : REAL, capacity : INT)
> - Status(id\* : CHAR, is_installed : INT, is_renting : INT, is_returning : INT, last_reported : CHAR, num_bikes_available : INT, num_bikes_available_ebike : INT, num_bikes_available_mechanical : INT, num_docks_available : INT)
> - La clé étrangère id de la relation Status fait référence à la clé primaire id de la relation Information.
>
> **Information :** `id` identifiant de la station ; `name` nom ; `capacity` nombre de bornettes ; `lat`, `lon` latitude et longitude.
> **Status :** `is_installed` station déployée (1/0) ; `is_renting` peut louer des vélos (1/0) ; `is_returning` peut recevoir des vélos (1/0) ; `last_reported` date de la dernière mise à jour ; `num_bikes_available` vélos disponibles (dont `..._ebike` électriques et `..._mechanical` mécaniques) ; `num_docks_available` bornettes disponibles.
>
> **Consignes.** Télécharger et exécuter le contenu du fichier velib.sql. Écrire des requêtes SQL **uniques** pour afficher les informations ci-dessous.

---
# MISE EN PLACE

En classe, les requêtes sont tapées dans la **console H2** (un SGBD). Ici, on utilise le module `sqlite3` de Python, qui contient un autre SGBD (**SQLite**) : **le langage SQL est le même**, donc chaque requête de ce notebook peut être recopiée telle quelle dans la console H2 (les rares différences sont signalées).

- `execute_fichier("fichier.sql")` joue le rôle de `RUNSCRIPT FROM 'fichier.sql';` dans H2 ;
- `affiche(requete)` exécute une requête et affiche le résultat sous forme de tableau, comme la console H2 ;
- `resultat(requete)` renvoie la liste des enregistrements : elle ne sert qu'aux vérifications.

La requête SQL est écrite entre trois guillemets `""" ... """` : c'est une chaîne de caractères Python qui peut tenir sur plusieurs lignes.

Chargement des données (dans H2 : `RUNSCRIPT FROM '/chemin/vers/velib.sql';`) :

In [1]:
import sqlite3

base = sqlite3.connect(":memory:")   # une base de données vide, en mémoire vive


def execute_fichier(nom_fichier):
    """Équivalent de RUNSCRIPT FROM '...' dans H2 : exécute un fichier .sql."""
    fichier = open(nom_fichier, encoding="utf-8")
    script = fichier.read()
    fichier.close()
    base.executescript(script)


def resultat(requete):
    """Renvoie la liste des enregistrements obtenus (pour les vérifications)."""
    return base.execute(requete).fetchall()


def affiche(requete, maximum=20):
    """Exécute une requête et affiche le résultat comme dans la console H2."""
    curseur = base.execute(requete)
    if curseur.description is None:          # INSERT, UPDATE, DELETE, CREATE...
        print("Instruction exécutée.")
        return
    entete = ""
    for colonne in curseur.description:
        entete = entete + colonne[0] + " | "
    print(entete)
    nombre = 0
    for ligne in curseur:
        if nombre < maximum:
            texte = ""
            for valeur in ligne:
                texte = texte + str(valeur) + " | "
            print(texte)
        nombre = nombre + 1
    if nombre > maximum:
        print("... (seules les " + str(maximum) + " premières lignes sont affichées)")
    print("(" + str(nombre) + " ligne(s))")


execute_fichier("velib.sql")
affiche("SELECT * FROM Information", 5)
affiche("SELECT * FROM Status", 5)

id | name | lat | lon | capacity | 
213688169 | Benjamin Godard - Victor Hugo | 48.865983 | 2.275725 | 35 | 
19179944124 | Hôpital Mondor | 48.798922410229 | 2.4537451531298 | 28 | 
18795078746 | Charcot - Benfleet | 48.878370277021 | 2.440523876268 | 28 | 
36255 | Toudouze - Clauzel | 48.87929591733507 | 2.3373600840568547 | 21 | 
251039991 | Cassini - Denfert-Rochereau | 48.837525839067 | 2.3360354080796 | 25 | 
... (seules les 5 premières lignes sont affichées)
(1522 ligne(s))
id | is_installed | is_renting | is_returning | last_reported | num_bikes_available | num_bikes_available_ebike | num_bikes_available_mechanical | num_docks_available | 
213688169 | 1 | 1 | 1 | 2026-11-14 02:00:52 | 17 | 14 | 3 | 17 | 
19179944124 | 1 | 1 | 1 | 2026-11-14 02:05:05 | 16 | 4 | 12 | 12 | 
18795078746 | 1 | 1 | 1 | 2026-11-14 02:00:32 | 10 | 9 | 1 | 18 | 
36255 | 1 | 1 | 1 | 2026-11-14 02:07:47 | 6 | 5 | 1 | 15 | 
251039991 | 1 | 1 | 1 | 2026-11-14 02:06:09 | 3 | 3 | 0 | 20 | 
... (seules les 5 pr

---
# QUESTION 1

#### 📝 ÉNONCÉ

> 1. Le nombre total de stations.

#### ✅ RÉSOLUTION

```sql
SELECT COUNT(*) FROM Information;
```

**Réponse :** 1 522 stations (avec cet instantané).

*Exécution dans le notebook :*

In [2]:
affiche("""
SELECT COUNT(*) FROM Information;
""")

COUNT(*) | 
1522 | 
(1 ligne(s))


*Vérification (non demandée) :*

In [3]:
assert resultat("SELECT COUNT(*) FROM Information") == [(1522,)]

#### 💡 EXPLICATION PÉDAGOGIQUE

- Chaque station a **un** enregistrement dans `Information` (son `id` est la clé primaire) : compter les enregistrements, c'est compter les stations.
- `SELECT COUNT(*) FROM Status;` donnerait ici le même nombre, mais c'est `Information` qui décrit les stations.

---
# QUESTION 2

#### 📝 ÉNONCÉ

> 2. Le nombre moyen de vélos électriques disponibles.

#### ✅ RÉSOLUTION

```sql
SELECT AVG(num_bikes_available_ebike) FROM Status;
```

**Réponse :** environ 5,03 vélos électriques par station.

*Exécution dans le notebook :*

In [4]:
affiche("""
SELECT AVG(num_bikes_available_ebike) FROM Status;
""")

AVG(num_bikes_available_ebike) | 
5.030880420499343 | 
(1 ligne(s))


*Vérification (non demandée) :*

In [5]:
moyenne = resultat("SELECT AVG(num_bikes_available_ebike) FROM Status")[0][0]
assert 5.0 < moyenne < 5.1

#### 💡 EXPLICATION PÉDAGOGIQUE

- `AVG` (*average*) calcule la **moyenne** d'un attribut sur tous les enregistrements sélectionnés.
- Le nombre de vélos électriques disponibles est un attribut de la relation `Status` (l'état de la station à un instant donné), et non de `Information` (les caractéristiques fixes de la station).

---
# QUESTION 3

#### 📝 ÉNONCÉ

> 3. Les noms des stations dont le nom contient le mot Lecourbe.

#### ✅ RÉSOLUTION

```sql
SELECT name FROM Information WHERE name LIKE '%Lecourbe%';
```

**Réponse :** 5 stations : Lecourbe - Jean Maridor, Charles Lecocq - Lecourbe, Sèvres - Lecourbe, Lecourbe - Convention, Volontaires - Lecourbe.

*Exécution dans le notebook :*

In [6]:
affiche("""
SELECT name FROM Information WHERE name LIKE '%Lecourbe%';
""")

name | 
Lecourbe - Jean Maridor | 
Charles Lecocq - Lecourbe | 
Sèvres - Lecourbe | 
Lecourbe - Convention | 
Volontaires - Lecourbe | 
(5 ligne(s))


*Vérification (non demandée) :*

In [7]:
assert len(resultat("SELECT name FROM Information WHERE name LIKE '%Lecourbe%'")) == 5

#### 💡 EXPLICATION PÉDAGOGIQUE

- « **contient** le mot » → `LIKE` avec `%` des deux côtés : `%` remplace **n'importe quelle suite de caractères** (même vide). `'%Lecourbe%'` trouve donc *Lecourbe* au début, au milieu ou à la fin du nom.
- ⚠️ `WHERE name = 'Lecourbe'` ne trouverait **rien** : aucune station ne s'appelle exactement « Lecourbe ».
- ⚠️ `'Lecourbe%'` ne trouverait que les noms qui **commencent** par Lecourbe (2 stations sur 5).
- Dans H2, `LIKE` distingue les majuscules : il faut écrire `Lecourbe` avec un L majuscule.

---
# QUESTION 4

#### 📝 ÉNONCÉ

> 4. Le nom de la station située la plus au sud.

#### ✅ RÉSOLUTION

```sql
SELECT name FROM Information
    WHERE lat = (SELECT MIN(lat) FROM Information);
```

**Réponse :** *Adrien Raynal - Verger* (avec ces données).

*Exécution dans le notebook :*

In [8]:
affiche("""
SELECT name FROM Information
    WHERE lat = (SELECT MIN(lat) FROM Information);
""")

name | 
Adrien Raynal - Verger | 
(1 ligne(s))


*Vérification (non demandée) :*

In [9]:
assert resultat("SELECT name FROM Information WHERE lat = (SELECT MIN(lat) FROM Information)") == [("Adrien Raynal - Verger",)]

#### 💡 EXPLICATION PÉDAGOGIQUE

- La plus au sud = la **plus petite latitude** `lat`.
- Le sujet demande une requête **unique** : on ne peut pas calculer `MIN(lat)` puis recopier la valeur à la main. On utilise une **requête imbriquée** (exemple page 8 du polycopié) : `(SELECT MIN(lat) FROM Information)` est calculée d'abord, puis utilisée comme une valeur dans le `WHERE`.
- ⚠️ `SELECT name, MIN(lat) FROM Information;` est **refusée** par H2 : on ne peut pas mélanger un attribut (une valeur par ligne) et une fonction d'agrégation (une seule valeur).

---
# QUESTION 5

#### 📝 ÉNONCÉ

> 5. Les noms et nombre de vélos disponibles des stations dont le nom contient le mot Lecourbe.

#### ✅ RÉSOLUTION

```sql
SELECT name, num_bikes_available FROM Information
    JOIN Status ON Information.id = Status.id
    WHERE name LIKE '%Lecourbe%';
```

*Exécution dans le notebook :*

In [10]:
affiche("""
SELECT name, num_bikes_available FROM Information
    JOIN Status ON Information.id = Status.id
    WHERE name LIKE '%Lecourbe%';
""")

name | num_bikes_available | 
Lecourbe - Jean Maridor | 15 | 
Charles Lecocq - Lecourbe | 24 | 
Sèvres - Lecourbe | 45 | 
Lecourbe - Convention | 28 | 
Volontaires - Lecourbe | 30 | 
(5 ligne(s))


*Vérification (non demandée) :*

In [11]:
assert len(resultat("""
SELECT name, num_bikes_available FROM Information
    JOIN Status ON Information.id = Status.id
    WHERE name LIKE '%Lecourbe%';
""")) == 5

#### 💡 EXPLICATION PÉDAGOGIQUE

- Le **nom** est dans `Information`, le **nombre de vélos** est dans `Status` : il faut une **jointure**. C'est l'exemple même du cours (page 7).
- La condition de jointure `Information.id = Status.id` relie chaque station à son état : `Status.id` est une **clé étrangère** qui fait référence à la clé primaire `Information.id`.
- Ici, les deux relations ont un attribut `id` : dans la condition de jointure, le préfixe `Information.` / `Status.` est **obligatoire**. Pour `name` et `num_bikes_available`, qui n'existent que dans une seule relation, il est facultatif.

---
# QUESTION 6

#### 📝 ÉNONCÉ

> 6. Le nombre de stations vides qui sont déployées et peuvent recevoir des vélos.

#### ✅ RÉSOLUTION

```sql
SELECT COUNT(*) FROM Status
    WHERE num_bikes_available = 0 AND is_installed = 1 AND is_returning = 1;
```

**Réponse :** 36 stations (avec cet instantané).

*Exécution dans le notebook :*

In [12]:
affiche("""
SELECT COUNT(*) FROM Status
    WHERE num_bikes_available = 0 AND is_installed = 1 AND is_returning = 1;
""")

COUNT(*) | 
36 | 
(1 ligne(s))


*Vérification (non demandée) :*

In [13]:
assert resultat("""
SELECT COUNT(*) FROM Status
    WHERE num_bikes_available = 0 AND is_installed = 1 AND is_returning = 1;
""") == [(36,)]

#### 💡 EXPLICATION PÉDAGOGIQUE

- On traduit chaque mot de la question en une condition :
  - « vides » → aucun vélo disponible : `num_bikes_available = 0` ;
  - « déployées » → `is_installed = 1` ;
  - « peuvent recevoir des vélos » → `is_returning = 1`.
- Les trois conditions doivent être vraies **en même temps** : on les relie par `AND`.
- Tous ces attributs sont dans `Status` : pas besoin de jointure, puisqu'on ne demande que le nombre.

---
# QUESTION 7

#### 📝 ÉNONCÉ

> 7. Le nom de la station avec le plus grand nombre de vélos disponibles.

#### ✅ RÉSOLUTION

```sql
SELECT name FROM Information
    JOIN Status ON Information.id = Status.id
    WHERE num_bikes_available = (SELECT MAX(num_bikes_available) FROM Status);
```

**Réponse :** *Route de Sèvres - Porte de Bagatelle*, avec 65 vélos (avec cet instantané).

*Exécution dans le notebook :*

In [14]:
affiche("""
SELECT name FROM Information
    JOIN Status ON Information.id = Status.id
    WHERE num_bikes_available = (SELECT MAX(num_bikes_available) FROM Status);
""")

name | 
Route de Sèvres - Porte de Bagatelle | 
(1 ligne(s))


*Vérification (non demandée) :*

In [15]:
assert resultat("""
SELECT name FROM Information
    JOIN Status ON Information.id = Status.id
    WHERE num_bikes_available = (SELECT MAX(num_bikes_available) FROM Status);
""") == [("Route de Sèvres - Porte de Bagatelle",)]

#### 💡 EXPLICATION PÉDAGOGIQUE

- C'est la combinaison des questions 4 et 5 : une **requête imbriquée** pour le maximum, et une **jointure** parce que le nom est dans `Information` alors que le nombre de vélos est dans `Status`.
- Si plusieurs stations avaient le même nombre maximal de vélos, elles seraient **toutes** affichées : la requête est juste dans tous les cas.

---
# QUESTION 8

#### 📝 ÉNONCÉ

> 8. Les noms et coordonnées GPS des stations pleines, dans l'ordre alphabétique.

#### ✅ RÉSOLUTION

```sql
SELECT name, lat, lon FROM Information
    JOIN Status ON Information.id = Status.id
    WHERE num_docks_available = 0
    ORDER BY name;
```

**Réponse :** 37 stations pleines (avec cet instantané).

*Exécution dans le notebook :*

In [16]:
affiche("""
SELECT name, lat, lon FROM Information
    JOIN Status ON Information.id = Status.id
    WHERE num_docks_available = 0
    ORDER BY name;
""")

name | lat | lon | 
Argenson - Château | 48.888559314669514 | 2.2642001509666447 | 
Aristide Briand - Place de la Résistance | 48.82124248401006 | 2.2511002421379094 | 
Charlot - Stade Gabriel Voisin | 48.827228120536 | 2.268122527474 | 
Convention - Lourmel | 48.842055041656 | 2.2859910875559 | 
Crozatier - Faubourg Saint-Antoine | 48.850440013405 | 2.378901366211 | 
Danielle Casanova - Ledru-Rollin | 48.815783497876815 | 2.383259721100331 | 
Enghien - Faubourg Poissonnière | 48.872420063053 | 2.3483952362828 | 
Faidherbe - Dahomey | 48.851358329771806 | 2.383823134795429 | 
Guersant - Gouvion-Saint-Cyr | 48.88287775178599 | 2.287667370814871 | 
Gustave Eiffel - Parc des Impressionnistes | 48.90390619651018 | 2.2946908324956894 | 
Haut du Mont-Mesly | 48.780191921646 | 2.4685737495183 | 
Jules Vallès - Charonne | 48.854284379935 | 2.3847674578428 | 
La Motte-Picquet - La Tour Maubourg | 48.856926701847 | 2.3096215831203 | 
Mairie de quartier du Plateau - Mont-Valérien | 48.8799933 | 2

*Vérification (non demandée) :*

In [17]:
assert len(resultat("""
SELECT name, lat, lon FROM Information
    JOIN Status ON Information.id = Status.id
    WHERE num_docks_available = 0
    ORDER BY name;
""")) == 37

#### 💡 EXPLICATION PÉDAGOGIQUE

- Une station est **pleine** quand on ne peut plus y rendre de vélo : il ne reste **aucune bornette libre**, `num_docks_available = 0`.
- ⚠️ « Pleine » ne se traduit pas par `num_bikes_available = capacity` : certaines bornettes peuvent être hors service, et la question 9 montre que le nombre de vélos peut même **dépasser** la capacité.
- Coordonnées GPS = `lat` et `lon` ; « ordre alphabétique » = `ORDER BY name`, placé **à la fin** de la requête.

---
# QUESTION 9

#### 📝 ÉNONCÉ

> 9. Les noms, nombres de vélos disponibles et capacités des stations dont le nombre de vélos disponibles est supérieur à la capacité, triés dans l'ordre décroissant de leurs capacités.

#### ✅ RÉSOLUTION

```sql
SELECT name, num_bikes_available, capacity FROM Information
    JOIN Status ON Information.id = Status.id
    WHERE num_bikes_available > capacity
    ORDER BY capacity DESC;
```

**Réponse :** 3 stations avec cet instantané : *BNF - Bibliothèque Nationale de France* (50 vélos pour 42 bornettes), *Westermeyer - Paul Vaillant-Couturier* (31 pour 25), *Sebastopol - Rambuteau* (18 pour 16).

*Exécution dans le notebook :*

In [18]:
affiche("""
SELECT name, num_bikes_available, capacity FROM Information
    JOIN Status ON Information.id = Status.id
    WHERE num_bikes_available > capacity
    ORDER BY capacity DESC;
""")

name | num_bikes_available | capacity | 
BNF - Bibliothèque Nationale de France | 50 | 42 | 
Westermeyer - Paul Vaillant-Couturier | 31 | 25 | 
Sebastopol - Rambuteau | 18 | 16 | 
(3 ligne(s))


*Vérification (non demandée) :*

In [19]:
assert resultat("""
SELECT name, num_bikes_available, capacity FROM Information
    JOIN Status ON Information.id = Status.id
    WHERE num_bikes_available > capacity
    ORDER BY capacity DESC;
""") == [("BNF - Bibliothèque Nationale de France", 50, 42),
         ("Westermeyer - Paul Vaillant-Couturier", 31, 25),
         ("Sebastopol - Rambuteau", 18, 16)]

#### 💡 EXPLICATION PÉDAGOGIQUE

- Dans le `WHERE`, on peut **comparer deux attributs** entre eux : `num_bikes_available > capacity` (le premier est dans `Status`, le second dans `Information`, d'où la jointure).
- `ORDER BY capacity DESC` : tri par capacité, **décroissant** (`DESC`, *descending*). Sans `DESC`, le tri serait croissant.
- **Comment est-ce possible ?** Plus de vélos que de bornettes ! Certaines stations acceptent des vélos « en surplus » (rattachés à la station sans être accrochés à une bornette), ou les données ne sont pas parfaitement à jour. Une base de données reflète le monde réel, avec ses imperfections : c'est justement ce que cette question fait découvrir.